# <font color='blue'> Appendix A4: Reward Models </font>

Reinforcement Learning from Human Feedback (RLHF) relies on a **Reward Model** to estimate how much humans prefer a generated response.

Instead of asking humans to assign numerical scores to every response,

the Reward Model is trained using **pairwise comparisons**.

Humans simply compare two candidate responses and indicate which one they prefer.

From many such comparisons,

the Reward Model learns to assign higher rewards to responses that humans consistently rank more highly.

This chapter explains

- why preference comparisons are used,
- how Reward Models are trained,
- and the mathematical foundations of preference learning.

---

# <font color='orange'> 1. Motivation </font>

Suppose the prompt is

```
Explain Newton's Second Law.
```

The language model generates two responses.

### Response A

```
Clear

Simple

Well Structured
```

### Response B

```
Technically Correct

Long

Hard to Read
```

Instead of assigning scores,

a human simply chooses

```
Response A
```

This single comparison contains enough information for training.

---

# <font color='orange'> 2. Why Pairwise Comparisons? </font>

Assigning numerical scores is surprisingly difficult.

Consider asking ten people

to rate a response

from

1

to

10.

Different people interpret the scale differently.

Some rarely give

10,

while others rarely give

5.

However,

asking

```
Which response do you prefer?
```

is much easier

and produces far more consistent data.

---

# <font color='orange'> 3. Reward Function </font>

The Reward Model is another neural network.

Given

- a prompt,
- a generated response,

it outputs

a scalar reward

$$
\boxed{
r(x,y),
}
$$

where

- \(x\) is the prompt,
- \(y\) is the response.

Higher rewards indicate

that humans are expected to prefer

the response.

Importantly,

the reward has **no absolute meaning**.

Only the **relative ordering** between responses matters.

---

# <font color='orange'> 4. Preference Dataset </font>

The training dataset consists of tuples

$$
\boxed{
(x,y_w,y_l),
}
$$

where

- \(x\) is the prompt,
- \(y_w\) is the preferred ("winning") response,
- \(y_l\) is the rejected ("losing") response.

Each comparison contributes one training example.

---

# <font color='orange'> 5. Bradley–Terry Model </font>

Suppose

two responses

have rewards

$$
r_w
\quad\text{and}\quad
r_l.
$$

The probability that

the preferred response

is chosen

is modelled as

$$
\boxed{
P(y_w \succ y_l)
=
\frac{e^{r_w}}
{e^{r_w}+e^{r_l}}.
}
$$

This is called

the **Bradley–Terry Model**.

Notice

only the **difference**

between the rewards

matters.

---

# <font color='orange'> 6. Intuition Behind the Bradley–Terry Model </font>

Suppose

```
Response A

Reward = 8
```

and

```
Response B

Reward = 2
```

Then

Response A

has a much larger probability

of being preferred.

If

both rewards

are similar,

both responses

have nearly equal probability.

Thus,

the Reward Model learns

relative rankings,

not absolute quality scores.

---

# <font color='orange'> 7. Binary Cross-Entropy Loss </font>

Suppose

the human preferred

Response A.

The Reward Model predicts

the probability

that this preference occurs.

The loss is

$$
\boxed{
L
=
-
\log
P(y_w \succ y_l).
}
$$

If

the Reward Model predicts

the human preference correctly,

the loss becomes small.

Otherwise,

the loss increases.

---

# <font color='orange'> 8. Learning Process </font>

The complete training pipeline is

```
Prompt

↓

Generate Two Responses

↓

Human Preference

↓

Reward Model

↓

Preference Probability

↓

Binary Cross-Entropy Loss

↓

Gradient Descent
```

Eventually,

the Reward Model learns

which types of responses

humans generally prefer.

---

# <font color='orange'> 9. What the Reward Model Learns </font>

The Reward Model gradually captures

preferences for

- clarity,
- correctness,
- politeness,
- helpfulness,
- conciseness,
- safety.

These properties are not explicitly programmed.

Instead,

they emerge

from the human comparison data.

---

# <font color='orange'> 10. Why Relative Rewards Matter </font>

Suppose

every reward

were increased by

100.

```
Old

↓

5

3
```

becomes

```
New

↓

105

103
```

The ranking remains unchanged.

Since only the **difference** between rewards affects the Bradley–Terry probability, the model does not need an absolute reward scale.

---

# <font color='orange'> 11. Connection to RLHF </font>

Once the Reward Model has been trained,

human comparisons

are no longer required

for every optimization step.

Instead,

the language model generates

a response,

the Reward Model predicts

its reward,

and PPO uses that reward

to improve the policy.

Thus,

the Reward Model acts as

a learned approximation

to human preferences.

---

# <font color='orange'> 12. Applications </font>

Reward Models are used in

- conversational AI,
- coding assistants,
- summarization,
- dialogue systems,
- preference optimization,
- Reinforcement Learning from Human Feedback.

They provide the reward signal needed for policy optimization.

---

# <font color='orange'> 13. Limitations </font>

Reward Models are only approximations

of human preferences.

If

the preference dataset

contains biases

or lacks diversity,

the Reward Model may learn those biases.

Furthermore,

if the language model learns to exploit weaknesses in the Reward Model,

it may generate responses that receive high predicted rewards without genuinely satisfying human intent.

This phenomenon is often called **reward hacking** or **reward model exploitation**.

---

# <font color='red'> 14. Mathematical Foundations </font>

Suppose

the Reward Model predicts

$$
r(x,y_w)
\quad\text{and}\quad
r(x,y_l).
$$

The Bradley–Terry probability is

$$
\boxed{
P(y_w\succ y_l)
=
\frac{
e^{r(x,y_w)}
}{
e^{r(x,y_w)}
+
e^{r(x,y_l)}
}.
}
$$

The binary cross-entropy loss becomes

$$
\boxed{
L
=
-
\log
P(y_w\succ y_l).
}
$$

Substituting the Bradley–Terry probability,

$$
\boxed{
L
=
-
\log
\left(
\frac{
e^{r(x,y_w)}
}{
e^{r(x,y_w)}
+
e^{r(x,y_l)}
}
\right).
}
$$

Using logarithm identities,

this is often rewritten as

$$
\boxed{
L
=
\log
\left(
1
+
e^{\,r(x,y_l)-r(x,y_w)}
\right).
}
$$

This expression has an intuitive interpretation:

- If the preferred response receives a much larger reward than the rejected response, the loss approaches zero.
- If the rejected response receives a larger reward, the loss becomes large, encouraging the model to reverse the ranking.

---

# <font color='orange'> 15. Reward Scores vs Preference Probabilities </font>

| Reward Score | Preference Probability |
|:---|:---|
| Scalar output of the Reward Model | Probability from the Bradley–Terry model |
| Relative quantity | Derived from reward differences |
| No absolute interpretation | Values lie between 0 and 1 |
| Used by PPO | Used during Reward Model training |

The Reward Model predicts scores,

while the Bradley–Terry model converts those scores into probabilities.

---

# <font color='orange'> 16. Common Misconceptions </font>

### Misconception 1

> The Reward Model predicts a true numerical measure of response quality.

**False.**

The reward values are relative. They are learned so that preferred responses receive higher scores than rejected ones, but the numerical values themselves have no intrinsic meaning.

---

### Misconception 2

> Humans provide reward values directly.

**False.**

Humans typically provide only preference comparisons. The Reward Model learns numerical rewards that are consistent with these rankings.

---

### Misconception 3

> A perfect Reward Model perfectly represents human preferences.

**False.**

The Reward Model is only an approximation learned from finite comparison data. It can inherit biases, make mistakes, or be exploited by the policy if it generalizes poorly.

---

# <font color='purple'> 17. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Reward Model | Neural network that predicts preference scores |
| Pairwise Comparison | Human chooses the better of two responses |
| Bradley–Terry Model | Converts reward differences into preference probabilities |
| Preference Dataset | Collection of prompt, preferred response, and rejected response |
| Binary Cross-Entropy | Loss function used to train the Reward Model |
| Relative Reward | Only reward differences matter |
| Reward Hacking | Policy exploits imperfections in the Reward Model |

> **Key Insight:** Reward Models translate human preference comparisons into a differentiable learning objective. Rather than assigning absolute quality scores, they learn relative reward values that preserve the ordering of responses preferred by humans. The Bradley–Terry model converts these rewards into preference probabilities, and binary cross-entropy trains the Reward Model to match observed human choices. Once trained, the Reward Model provides the reward signal used during Reinforcement Learning from Human Feedback, allowing the language model to improve without requiring human evaluation of every generated response.